# Phase II: Data Pipeline (Kyle)

This notebook covers **rubric Part 2** (obtain, clean, and merge all data) and provides the **pipeline summary** for Part 1. The finished cells get copied into the group's main notebook.

### How the group uses this notebook

The free API plan allows 100 requests per person per day, and each fighter's record costs one request. To collect more data, **each group member runs this notebook once with their own free API key** and a different `MY_BATCH` number (1, 2, or 3). Each person's results are saved to their own file (`records_batch1.json`, etc.). Everyone uploads their file to the GitHub repo, and Kyle runs the merge section to combine them.

**Before teammates run it:** Kyle uploads `fighters_raw.json` to the repo. Teammates put it in the same folder as this notebook so they don't spend requests re-downloading the fighter list.

## Pipeline summary (for Part 1 of the group notebook)

Data comes from **API-MMA** (API-Sports, `https://v1.mma.api-sports.io/`), which returns JSON. The pipeline has five steps:

1. **Weight classes** (`/categories`, 1 request): returns 18 category names, including non-standard ones like "Catch Weight" and "Open Weight".
2. **Fighter profiles** (`/fighters?category=...`, 1 request per category, 18 total): returns about 2,500 fighter records with name, gender, height, weight, reach, stance, and weight class. Saved to `fighters_raw.json`.
3. **Cleaning the profiles:** height (`5' 9'`), reach (`71'`), and weight (`136 lbs`) arrive as text and are converted to numbers in inches and pounds. Fighters listed only under non-standard categories are removed, and each fighter is kept once under their standard weight class. Fighters missing height, reach, or stance are excluded from sampling.
4. **Fighter records** (`/fighters/records?id=...`, 1 request per fighter): a fixed random sample of fighters (same `random_state` for everyone) is split into batches, and each group member downloads one batch with their own API key. Each response gives total wins, losses, and draws, plus wins by KO and by submission. Results are saved after every request, so an interrupted run resumes without wasting requests.
5. **Merging and feature engineering:** all batch files are combined and joined to the profiles on fighter `id`. Derived columns: `decision_wins` (wins − KO wins − submission wins), `win_pct` (wins ÷ (wins + losses)), `win_style` (the most common way the fighter wins), and `reach_vs_class` (reach minus the median reach of the fighter's weight class). The result is saved to `ufc_fighters_clean.csv`.

## 1. Setup

In [ ]:
import os
import json
import time
import requests
import pandas as pd
from getpass import getpass

MY_BATCH = 1        # Kyle = 1, second member = 2, third member = 3
BATCH_SIZE = 75     # fighters per person (75 records + a few other calls stays under 100/day)

API_KEY = getpass("Paste your API key and press Enter: ").strip()
BASE_URL = "https://v1.mma.api-sports.io/"
HEADERS = {"x-apisports-key": API_KEY}


def get(endpoint, params=None):
    """Call the API and return the JSON response as a Python dictionary."""
    response = requests.get(BASE_URL + endpoint, headers=HEADERS, params=params)
    data = response.json()
    if data.get("errors"):
        raise Exception(f"API error: {data['errors']}")
    return data


# /status is free. It shows how many requests you've used today.
print(get("status")["response"]["requests"])

## 2. Fighter profiles

Loads `fighters_raw.json` if it's in the folder (teammates should get it from the GitHub repo). Otherwise downloads it, which costs 19 requests.

In [ ]:
if os.path.exists("fighters_raw.json"):
    with open("fighters_raw.json") as f:
        fighters_raw = json.load(f)
    print("Loaded fighters from file.")
else:
    categories = get("categories")["response"]
    fighters_raw = []
    for category in categories:
        data = get("fighters", {"category": category})
        fighters_raw.extend(data["response"])
        print(category, ":", len(data["response"]), "fighters")
        time.sleep(7)  # free plan allows about 10 requests per minute
    with open("fighters_raw.json", "w") as f:
        json.dump(fighters_raw, f)

print("Total fighter entries:", len(fighters_raw))

## 3. Clean the fighter profiles

* Convert height, reach, and weight from text to numbers. The API marks inches with a single quote (`5' 9'`, `71'`).
* Remove non-standard weight categories. A fighter can be listed under both "Catch Weight" and their real division, and catchweight bouts don't tell us which division a fighter belongs to. Women's Lightweight and Super Heavyweight are also removed because they have only 1–2 fighters.
* Keep each fighter once.

In [ ]:
def height_to_inches(text):
    """Convert a height like 5' 9' or 5' 9" to inches (5 * 12 + 9 = 69)."""
    if not text:
        return None
    parts = str(text).replace("'", " ").replace('"', " ").split()
    try:
        if len(parts) == 2:
            return int(parts[0]) * 12 + float(parts[1])
        if len(parts) == 1:
            return int(parts[0]) * 12
    except ValueError:
        return None
    return None


def text_to_number(text):
    """Convert text like 71' or 136 lbs to a plain number (71.0 or 136.0)."""
    if not text:
        return None
    text = str(text).replace("'", " ").replace('"', " ")
    try:
        return float(text.split()[0])
    except (ValueError, IndexError):
        return None


NON_STANDARD = ["Catch Weight", "Catchweight", "Open Weight", "Super Heavyweight",
                "Women's Catch Weight", "Women's Lightweight"]

rows = []
for f in fighters_raw:
    if f.get("category") in NON_STANDARD:
        continue
    rows.append({
        "id": f.get("id"),
        "name": f.get("name"),
        "gender": f.get("gender"),
        "weight_class": f.get("category"),
        "stance": f.get("stance"),
        "height_in": height_to_inches(f.get("height")),
        "weight_lbs": text_to_number(f.get("weight")),
        "reach_in": text_to_number(f.get("reach")),
    })

fighters = pd.DataFrame(rows).drop_duplicates("id")
print("Fighters after cleaning:", len(fighters))
print("Missing values:")
print(fighters.isna().sum())
fighters.head()

## 4. Download records for my batch

Everyone gets the **same** shuffled list of fighters (because `random_state` is fixed), then each person takes their own slice:
batch 1 = fighters 1–75, batch 2 = fighters 76–150, batch 3 = fighters 151–225.

If the daily limit is reached, the loop stops and keeps everything collected so far. Re-running the cell the next day continues where it left off.

In [ ]:
known = fighters.dropna(subset=["height_in", "reach_in", "stance"])
shuffled = known.sample(frac=1, random_state=3000)   # same order for everyone

start = (MY_BATCH - 1) * BATCH_SIZE
my_fighters = shuffled.iloc[start : start + BATCH_SIZE]
print(f"Fighters with full physical data: {len(known)}")
print(f"Batch {MY_BATCH}: fighters {start + 1} to {start + len(my_fighters)}")

batch_file = f"records_batch{MY_BATCH}.json"
if os.path.exists(batch_file):
    with open(batch_file) as f:
        my_records = json.load(f)
else:
    my_records = {}

for fighter_id in my_fighters["id"]:
    if str(fighter_id) in my_records:
        continue  # already downloaded
    try:
        data = get("fighters/records", {"id": fighter_id})
    except Exception as error:
        print("Stopped early:", error)
        break
    if data["response"]:
        my_records[str(fighter_id)] = data["response"][0]
    with open(batch_file, "w") as f:
        json.dump(my_records, f)
    time.sleep(7)

print(f"Records in {batch_file}: {len(my_records)}")

## 5. Merge all batches (Kyle runs this after collecting everyone's files)

Put every `records_batch*.json` file from the GitHub repo in this folder. If you still have `records_raw.json` from the proposal, it's included too. Duplicate fighters are only counted once.

In [ ]:
all_records = {}
for filename in sorted(os.listdir(".")):
    if filename.startswith("records_") and filename.endswith(".json"):
        with open(filename) as f:
            batch = json.load(f)
        all_records.update(batch)
        print(f"{filename}: {len(batch)} records")

print("Total unique fighters with records:", len(all_records))

record_rows = []
for fighter_id, r in all_records.items():
    record_rows.append({
        "id": int(fighter_id),
        "wins": r["total"]["win"],
        "losses": r["total"]["loss"],
        "draws": r["total"]["draw"],
        "ko_wins": r["ko"]["win"],
        "sub_wins": r["sub"]["win"],
    })
records = pd.DataFrame(record_rows)

df = fighters.merge(records, on="id", how="inner")
print("Rows after merging profiles with records:", len(df))

## 6. Feature engineering and quality checks

In [ ]:
df["decision_wins"] = df["wins"] - df["ko_wins"] - df["sub_wins"]
df["total_fights"] = df["wins"] + df["losses"] + df["draws"]
df["win_pct"] = df["wins"] / (df["wins"] + df["losses"])


def win_style(row):
    if row["ko_wins"] >= row["sub_wins"] and row["ko_wins"] >= row["decision_wins"]:
        return "KO"
    elif row["sub_wins"] >= row["decision_wins"]:
        return "Submission"
    else:
        return "Decision"


df["win_style"] = df.apply(win_style, axis=1)
df["reach_vs_class"] = df["reach_in"] - df.groupby("weight_class")["reach_in"].transform("median")

# Quality checks: remove rows that can't be right
before = len(df)
df = df[df["total_fights"] > 0]                  # no fights recorded
df = df[df["decision_wins"] >= 0]                # KO + sub wins can't exceed total wins
df = df[df["height_in"].between(58, 84)]         # 4'10" to 7'0"
df = df[df["reach_in"].between(55, 90)]          # realistic arm spans
print(f"Removed {before - len(df)} rows that failed quality checks")

df = df.reset_index(drop=True)
df.to_csv("ufc_fighters_clean.csv", index=False)
print("Final data set:", df.shape)
df.head(10)

In [ ]:
print("Fighters per weight class:")
print(df["weight_class"].value_counts())
print()
print("Fighters per stance:")
print(df["stance"].value_counts())
print()
print("Main way of winning:")
print(df["win_style"].value_counts())
print()
print("Gender:")
print(df["gender"].value_counts())

In [ ]:
df[["height_in", "reach_in", "reach_vs_class", "wins", "losses", "total_fights", "win_pct"]].describe().round(2)